### Usando RNNs para classificar sentimento em dados do IMDB
Neste exercício, treinaremos uma RNN "vanilla" para prever o sentimento em avaliações do IMDB. Nossos dados consistem em 25000 sequências de treinamento e 25000 sequências de teste. O resultado é binário (positivo/negativo) e ambos os resultados são igualmente representados tanto no conjunto de treinamento quanto no conjunto de teste.

O Keras fornece uma interface conveniente para carregar os dados e codificar imediatamente as palavras em inteiros (com base nas palavras mais comuns). Isso nos poupará muito do trabalho pesado que geralmente está envolvido ao trabalhar com texto bruto.

Vamos percorrer a preparação dos dados e a construção de um modelo RNN. Então será sua vez de construir seus próprios modelos (e preparar os dados como achar melhor).

In [15]:
from __future__ import print_function
import tensorflow.keras as keras
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Embedding
from tensorflow.keras.layers import SimpleRNN
from tensorflow.keras.datasets import imdb
from tensorflow.keras.optimizers import RMSprop
from tensorflow.keras import initializers

In [2]:
max_features = 20000  # Usado no carregamento dos dados, seleciona as palavras mais comuns (max_features)
maxlen = 30  # comprimento máximo de uma sequência - truncar após isso
batch_size = 32

In [3]:
## Carregar os dados. A função tokeniza automaticamente o texto em inteiros distintos
(x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
print(len(x_train), 'sequências de treino')
print(len(x_test), 'sequências de teste')

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
25000 sequências de treino
25000 sequências de teste


In [4]:
# Isso preenche (ou trunca) as sequências para que tenham o comprimento máximo
x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
print('forma de x_train:', x_train.shape)
print('forma de x_test:', x_test.shape)

forma de x_train: (25000, 30)
forma de x_test: (25000, 30)


In [5]:
x_train[123,:]  # Aqui está como uma sequência de exemplo se parece

array([  219,   141,    35,   221,   956,    54,    13,    16,    11,
        2714,    61,   322,   423,    12,    38,    76,    59,  1803,
          72,     8, 10508,    23,     5,   967,    12,    38,    85,
          62,   358,    99], dtype=int32)

## Camadas Keras para RNNs (Vanilla)

Neste exercício, não usaremos vetores de palavras pré-treinados. Em vez disso, aprenderemos um embedding como parte da Rede Neural. Isso é representado pela Camada de Embedding abaixo.

### Camada de Embedding
`keras.layers.embeddings.Embedding(input_dim, output_dim, embeddings_initializer='uniform', embeddings_regularizer=None, activity_regularizer=None, embeddings_constraint=None, mask_zero=False, input_length=None)`

- Esta camada mapeia cada inteiro em um vetor de palavra (denso) distinto de comprimento `output_dim`.
- Pode-se pensar nisso como aprender um embedding de vetor de palavra "em tempo real" em vez de usar um mapeamento existente (como GloVe)
- O `input_dim` deve ser o tamanho do vocabulário.
- O `input_length` especifica o comprimento das sequências que a rede espera.

### Camada SimpleRNN
`keras.layers.recurrent.SimpleRNN(units, activation='tanh', use_bias=True, kernel_initializer='glorot_uniform', recurrent_initializer='orthogonal', bias_initializer='zeros', kernel_regularizer=None, recurrent_regularizer=None, bias_regularizer=None, activity_regularizer=None, kernel_constraint=None, recurrent_constraint=None, bias_constraint=None, dropout=0.0, recurrent_dropout=0.0)`

- Esta é a RNN básica, onde a saída também é alimentada de volta como o "estado oculto" para a próxima iteração.
- O parâmetro `units` fornece a dimensionalidade da saída (e, portanto, do estado oculto). Note que tipicamente haverá outra camada após a RNN mapeando a saída (RNN) para a saída da rede. Então devemos pensar neste valor como a dimensionalidade desejada do estado oculto e não necessariamente a saída desejada da rede.
- Lembre-se de que existem dois conjuntos de pesos, um para a fase "recorrente" e outro para a fase "kernel". Eles podem ser configurados separadamente em termos de sua inicialização, regularização, etc.






In [7]:
## Vamos construir uma RNN

rnn_hidden_dim = 5
word_embedding_dim = 50
model_rnn = Sequential()
model_rnn.add(Embedding(max_features, word_embedding_dim))  # Esta camada pega cada inteiro na sequência e o incorpora em um vetor de 50 dimensões
model_rnn.add(SimpleRNN(rnn_hidden_dim,
                    kernel_initializer=initializers.RandomNormal(stddev=0.001),
                    recurrent_initializer=initializers.Identity(gain=1.0),
                    activation='relu',
                    input_shape=x_train.shape[1:]))

model_rnn.add(Dense(1, activation='sigmoid'))

/usr/local/lib/python3.13/dist-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [8]:
## Note que a maioria dos parâmetros vem da camada de embedding
model_rnn.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [10]:
rmsprop = keras.optimizers.RMSprop(learning_rate=.0001)

model_rnn.compile(loss='binary_crossentropy',
              optimizer=rmsprop,
              metrics=['accuracy'])

(25000, 30)
[  18   51   36   28  224   92   25  104    4  226   65   16   38 1334
   88   12   16  283    5   16 4472  113  103   32   15   16 5345   19
  178   32]
[0 0 0 0 0 0 0 0 0 0]


In [11]:
model_rnn.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=10,
          validation_data=(x_test, y_test))

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 16ms/step - accuracy: 0.5966 - loss: 0.6669 - val_accuracy: 0.6674 - val_loss: 0.6156
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 7ms/step - accuracy: 0.7068 - loss: 0.5681 - val_accuracy: 0.7250 - val_loss: 0.5411
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7579 - loss: 0.4980 - val_accuracy: 0.7502 - val_loss: 0.5037
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7860 - loss: 0.4545 - val_accuracy: 0.7638 - val_loss: 0.4805
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8070 - loss: 0.4251 - val_accuracy: 0.7702 - val_loss: 0.4693
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8173 - loss: 0.4046 - val_accuracy: 0.7768 - val_loss: 0.4621
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8266 - loss: 0.3897 - val_accuracy: 0.7804 - val_loss: 0.4553
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 7ms/step - accuracy: 0.8311 - loss: 0.3781 - val_accuracy:

In [ ]:
score, acc = model_rnn.evaluate(x_test, y_test,
                            batch_size=batch_size)
print('Pontuação no teste:', score)
print('Precisão no teste:', acc)

782/782 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - accuracy: 0.7860 - loss: 0.4519
Pontuação no teste: 0.4518597722053528
Precisão no teste: 0.7859600186347961


## Exercício
### Sua Vez

Agora faça você mesmo:
- Prepare os dados para usar sequências de comprimento 80 em vez de comprimento 30. Isso melhorou o desempenho?
- Tente diferentes valores de "max_features". Você consegue melhorar o desempenho?
- Tente tamanhos menores e maiores da dimensão oculta da RNN. Como isso afeta o desempenho do modelo? Como afeta o tempo de execução?

**modelo 1:**

Pontuação no teste: 0.4518597722053528<br>
Precisão no teste: 0.7859600186347961<br>
accuracy: 0.8418 - loss: 0.3616 - val_accuracy: 0.7860 - val_loss: 0.4526<br>

**modelo 2 (max_len = 80):**

Pontuação no teste: 0.3806734085083008<br>
Precisão no teste: 0.830079972743988<br>
accuracy: 0.8891 - loss: 0.2712 - val_accuracy: 0.8301 - val_loss: 0.3807<br>

Se saiu melhor, pega mais das avaliações!

**modelo 3(outros max_features):**

Acho que vai dar overfitting, vai aumentar mt os pesos do embedding

Pontuação no teste: 0.3856298327445984<br>
Precisão no teste: 0.8291599750518799<br>
accuracy: 0.8886 - loss: 0.2747 - val_accuracy: 0.8292 - val_loss: 0.3856<br>

nao deu um resultado mt diferente do modelo 2, so aumentou um pouco a diferença
das metricas e a validação, e a acurácia no teste ficou menor ainda. acredito que aumentar so vai piorar.

**modelo 4(outros max_features):**

Pontuação no teste: 0.42002347111701965<br>
Precisão no teste: 0.8065999746322632<br>
accuracy: 0.8608 - loss: 0.3251 - val_accuracy: 0.8066 - val_loss: 0.4200<br>

esse resultado de epoca é sempre os da 10, e a diferença da acurácia e loss para as de validação aumentaram, isso indica overfitting, so piora o modelo, vou tentar diminuir

**modelo 5(outros max_features):**


Pontuação no teste: 0.3876306116580963<br>
Precisão no teste: 0.8268399834632874<br>
accuracy: 0.8793 - loss: 0.2922 - val_accuracy: 0.8268 - val_loss: 0.3876<br>

Não mudou quase nada... usando o mesmo maxlen do modelo 2
emfim, não deu pra mudar pra melhor

sobre os testes com os rnn dims, não teve grande efeito na acurácia, mas aumentou a distancia do treino e validação, além de aumentar o tempo para rodar. Só deu overfitting

In [22]:
# Escreva seu código aqui

def treino_completo(max_features=20000, maxlen=30, rnn_hidden_dim=5, word_embedding_dim=50, batch_size=32):
  (x_train, y_train), (x_test, y_test) = imdb.load_data(num_words=max_features)
  x_train = sequence.pad_sequences(x_train, maxlen=maxlen)
  x_test = sequence.pad_sequences(x_test, maxlen=maxlen)
  #separando os dados

  rnn = Sequential()
  rnn.add(Embedding(max_features, word_embedding_dim))
  rnn.add(SimpleRNN(rnn_hidden_dim,
                    kernel_initializer=initializers.RandomNormal(stddev=0.001),
                    recurrent_initializer=initializers.Identity(gain=1.0),
                    activation='relu',))
  """isso do U quase 0 e o W identidade pra manter contexto
  explodiu minha mente, muito obrigado por transmitir tal sabedoria"""
  rnn.add(Dense(1, activation='sigmoid'))
  #modelo rnn

  rnn.compile(loss='binary_crossentropy',
              optimizer=RMSprop(learning_rate=.0001),
              metrics=['accuracy'])

  rnn.fit(x_train, y_train,
          batch_size=batch_size,
          epochs=10,
          validation_data=(x_test, y_test))

  print("\n")

  score, acc = rnn.evaluate(x_test, y_test,
                            batch_size=batch_size)
  print('Pontuação no teste:', score)
  print('Precisão no teste:', acc)


In [23]:
#modelo 2

treino_completo(maxlen=80)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 19s 17ms/step - accuracy: 0.5972 - loss: 0.6567 - val_accuracy: 0.7053 - val_loss: 0.5727
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 11s 14ms/step - accuracy: 0.7558 - loss: 0.5068 - val_accuracy: 0.7738 - val_loss: 0.4816
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8038 - loss: 0.4277 - val_accuracy: 0.7700 - val_loss: 0.4615
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8327 - loss: 0.3801 - val_accuracy: 0.8092 - val_loss: 0.4186
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8483 - loss: 0.3477 - val_accuracy: 0.8183 - val_loss: 0.4011
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8606 - loss: 0.3259 - val_accuracy: 0.8188 - val_loss: 0.3943
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8706 - loss: 0.3083 - val_accuracy: 0.8214 - val_loss: 0.3926
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8777 - loss: 0.2946 - val_ac

In [24]:
#modelo 3

treino_completo(max_features=25000, maxlen=80)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 20s 19ms/step - accuracy: 0.6048 - loss: 0.6543 - val_accuracy: 0.6622 - val_loss: 0.6073
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7438 - loss: 0.5218 - val_accuracy: 0.7308 - val_loss: 0.5316
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.7918 - loss: 0.4484 - val_accuracy: 0.7706 - val_loss: 0.4721
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8220 - loss: 0.4005 - val_accuracy: 0.7880 - val_loss: 0.4454
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8420 - loss: 0.3651 - val_accuracy: 0.7996 - val_loss: 0.4311
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8566 - loss: 0.3375 - val_accuracy: 0.8126 - val_loss: 0.4084
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8689 - loss: 0.3162 - val_accuracy: 0.8152 - val_loss: 0.4063
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8765 - loss: 0.3000 - val_ac

In [25]:
# modelo 4

treino_completo(max_features=30000, maxlen=40)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 16ms/step - accuracy: 0.6108 - loss: 0.6575 - val_accuracy: 0.6957 - val_loss: 0.5952
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.7256 - loss: 0.5516 - val_accuracy: 0.7385 - val_loss: 0.5258
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.7706 - loss: 0.4815 - val_accuracy: 0.7617 - val_loss: 0.4849
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.7996 - loss: 0.4363 - val_accuracy: 0.7802 - val_loss: 0.4577
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8184 - loss: 0.4041 - val_accuracy: 0.7914 - val_loss: 0.4411
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8322 - loss: 0.3800 - val_accuracy: 0.7962 - val_loss: 0.4330
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8402 - loss: 0.3616 - val_accuracy: 0.8003 - val_loss: 0.4254
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 7s 9ms/step - accuracy: 0.8496 - loss: 0.3469 - val_accuracy: 

In [26]:
#modelo 5

treino_completo(max_features=15000, maxlen=80)

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.6089 - loss: 0.6626 - val_accuracy: 0.7003 - val_loss: 0.5926
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7419 - loss: 0.5262 - val_accuracy: 0.7330 - val_loss: 0.5218
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7893 - loss: 0.4523 - val_accuracy: 0.7723 - val_loss: 0.4709
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8144 - loss: 0.4097 - val_accuracy: 0.7879 - val_loss: 0.4467
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8342 - loss: 0.3783 - val_accuracy: 0.7982 - val_loss: 0.4283
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8479 - loss: 0.3533 - val_accuracy: 0.8066 - val_loss: 0.4183
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 10ms/step - accuracy: 0.8582 - loss: 0.3336 - val_accuracy: 0.8151 - val_loss: 0.4009
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8653 - loss: 0.3174 - val_acc

In [30]:
#ultimos modelos

import time

for units in [5, 16, 32, 64, 128]:
    t0 = time.time()
    treino_completo(rnn_hidden_dim=units, maxlen=80)
    print(f'\nhidden_dim = {units}')
    print(f'tempo total: {time.time() - t0:.0f}s')
    print("\n")

Epoch 1/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 18s 17ms/step - accuracy: 0.5620 - loss: 0.6819 - val_accuracy: 0.6168 - val_loss: 0.6468
Epoch 2/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7211 - loss: 0.5549 - val_accuracy: 0.7422 - val_loss: 0.5247
Epoch 3/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7837 - loss: 0.4613 - val_accuracy: 0.7589 - val_loss: 0.4982
Epoch 4/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 8s 11ms/step - accuracy: 0.8164 - loss: 0.4089 - val_accuracy: 0.7924 - val_loss: 0.4417
Epoch 5/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8393 - loss: 0.3718 - val_accuracy: 0.8052 - val_loss: 0.4263
Epoch 6/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8521 - loss: 0.3466 - val_accuracy: 0.8034 - val_loss: 0.4260
Epoch 7/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 10s 11ms/step - accuracy: 0.8620 - loss: 0.3278 - val_accuracy: 0.8180 - val_loss: 0.4031
Epoch 8/10
782/782 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.8688 - loss: 0.3116 - val_ac